# WACV 2027 Workshop on Real-World Surveillance (RWS) — Track 2: ABPR
## EXP-007: Championship Cross-Modal Metric Learning System (Dual-GPU DDP)

> **Target Hardware:** Kaggle Dual Tesla T4 GPUs (32GB Total VRAM, 30GB CPU RAM)
> **Paradigm:** Continuous Hyperspherical Metric Learning in $\mathbb{R}^{1024}$ (Swin-T + ArcFace + SDM).
> **Objective:** Completely eliminate the 40-dimensional probability bottleneck that caused the 0.40–0.45 ceiling across all prior experiments. Maps images and natural language pseudo-descriptions into a shared 1024-D hypersphere using Additive Angular Margin (ArcFace) and Similarity Distribution Matching (SDM) losses.
> **Two-Stage Execution:**
>   - **Stage 1 (30 Epochs):** Visual representation warmup with Surveillance Degradation Transforms (75px downscaling + motion blur + random erasing).
>   - **Stage 2 (25 Epochs):** Cross-modal metric margin alignment with frozen/differentially-tuned Swin-T visual backbone and dual-head query adapters.
> **Submission Output:** Self-contained, air-gapped  (<60 MB, zero external dependencies, $<9$ min CPU ingestion runtime).


In [ ]:
# ==============================================================================
# 1. HARDWARE TELEMETRY & SYSTEM CAPABILITIES AUDIT
# ==============================================================================
import os, sys, psutil, torch

print("=" * 80)
print("KAGGLE SYSTEM HARDWARE AUDIT")
print("=" * 80)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Detected CUDA GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  -> GPU {i}: {p.name} | Total VRAM: {p.total_memory / (1024**3):.2f} GB")

mem = psutil.virtual_memory()
print(f"Host System RAM: {mem.total / (1024**3):.2f} GB (Available: {mem.available / (1024**3):.2f} GB)")
print(f"CPU Count: {psutil.cpu_count(logical=True)} logical cores")
print("=" * 80)
assert torch.cuda.is_available(), "CUDA GPU acceleration required! Please enable GPU in Kaggle settings."

In [ ]:
# ==============================================================================
# 2. DATASET LOCATION & AUTO-DETECTION
# ==============================================================================
import os, glob

def detect_dataset_root():
    candidates = [
        "/kaggle/input/datasets/hinewk/upar-challenge-2027-data",
        "/kaggle/input/upar-challenge-2027-data",
        "/kaggle/input/upar-challenge-2027",
        "/kaggle/input/wacv2027-track2",
        "data",
        "UPAR-Challenge-2027/data"
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "annotations", "train.csv")):
            return os.path.abspath(c)
        sub = os.path.join(c, "data")
        if os.path.exists(os.path.join(sub, "annotations", "train.csv")):
            return os.path.abspath(sub)
    
    # Deep search
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train.csv" in files and os.path.basename(root) == "annotations":
            return os.path.abspath(os.path.dirname(root))
    return None

DATA_DIR = detect_dataset_root()
print(f"[DATASET] Resolved Dataset Root: {DATA_DIR}")
assert DATA_DIR is not None, "Failed to find annotations/train.csv! Verify Kaggle dataset attachment."

In [ ]:
# ==============================================================================
# 3. GITHUB REPOSITORY INGESTION (Auto-Clone / Pull)
# ==============================================================================
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = "YOUR_GITHUB_PAT_TOKEN_HERE"

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git"
REPO_DIR = "/kaggle/working/upar_challenge_26_rws27"

if os.path.exists(REPO_DIR):
    print("Repository directory exists. Pulling latest commits from main...")
    os.system(f"cd {REPO_DIR} && git fetch origin && git reset --hard origin/main")
else:
    print("Cloning private repository from GitHub...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print(f"Active Directory: {os.getcwd()}")


In [ ]:
# ==============================================================================
# 4. LAUNCH DUAL-GPU DDP TRAINING (TORCHRUN NCCL)
# ==============================================================================
# 2x Tesla T4 GPUs with NCCL backend and FP16 Autocast
!cd /kaggle/working/upar_challenge_26_rws27 && \
PYTHONUNBUFFERED=1 \
OMP_NUM_THREADS=1 \
PYTHONPATH=/kaggle/working/upar_challenge_26_rws27:. \
torchrun --nproc_per_node=2 championship_pipeline/train.py \
    --data_root "$DATA_DIR" \
    --save_dir "/kaggle/working/checkpoints/EXP-007" \
    --exp_id "EXP-007" \
    --batch_size 128 \
    --val_batch_size 128 \
    --stage1_epochs 30 \
    --stage2_epochs 25 \
    --lr 3.5e-4 \
    --backbone_lr_stage1 1.0e-4 \
    --backbone_lr_stage2 1.0e-6 \
    --num_workers 2 \
    --eval_interval 1


In [ ]:
# ==============================================================================
# 5. CODABENCH SUBMISSION VERIFICATION & ARCHIVE INSPECTION
# ==============================================================================
import os, zipfile, tempfile, sys, numpy as np
from PIL import Image

candidate_zips = [
    "submissions/EXP-007_submission.zip",
    "/kaggle/working/submissions/EXP-007_submission.zip",
    "/kaggle/working/upar_challenge_26_rws27/submissions/EXP-007_submission.zip"
]
sub_zip = None
for cz in candidate_zips:
    if os.path.exists(cz):
        sub_zip = cz
        break

if sub_zip is None:
    print("Warning: EXP-007_submission.zip not found yet. Check training logs above.")
else:
    zip_mb = os.path.getsize(sub_zip) / (1024 * 1024)
    print(f"Found submission archive: {sub_zip} ({zip_mb:.2f} MB)")
    assert zip_mb < 200.0, f"Size {zip_mb:.2f} MB exceeds Codabench 200 MB limit!"
    
    with tempfile.TemporaryDirectory() as td:
        with zipfile.ZipFile(sub_zip, "r") as z:
            z.extractall(td)
            print("Archive contents:", z.namelist())
        
        gal_dir = os.path.join(td, "mock_gallery")
        os.makedirs(gal_dir, exist_ok=True)
        gal_items = []
        for i in range(10):
            p = os.path.join(gal_dir, f"sample_{i}.jpg")
            Image.new("RGB", (128, 256), color=(i*20, 100, 150)).save(p)
            gal_items.append({"image_path": p})
        
        mock_sample = {
            "gallery": gal_items,
            "queries": np.random.randint(0, 2, (5, 40)).astype(np.float32),
            "attribute_names": [f"attr_{i}" for i in range(40)]
        }
        
        sys.path.insert(0, td)
        import run
        res = run.rank_gallery(mock_sample)
        dists = res["distances"]
        print(f"Simulated rank_gallery returned distance matrix: shape={dists.shape}")
        assert dists.shape == (5, 10), f"Unexpected shape: {dists.shape}"
        assert np.all(np.isfinite(dists)), "Found non-finite values in distance matrix!"
        print("=" * 80)
        print("CODABENCH INGESTION SIMULATION 100% PASSED!")
        print(f"Ready for upload to Codabench: {sub_zip}")
        print("=" * 80)